In [15]:
# %pip install transformers torch --quiet

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict, Tuple


In [16]:
class ToxicityDetector:
    """
    RoBERTa-based multi-class toxicity detector.

    Labels:
        0 -> friendly
        1 -> neutral
        2 -> rude
        3 -> toxic
        4 -> super_toxic
    """

    _instance = None  # Singleton cache

    LABELS = [
        "friendly",
        "neutral",
        "rude",
        "toxic",
        "super_toxic"
    ]

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)

            model_name = "roberta-base"  # Replace with fine-tuned path if available
            cls._instance.tokenizer = AutoTokenizer.from_pretrained(model_name)
            cls._instance.model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=len(cls.LABELS)
            )

            cls._instance.device = torch.device(
                "cuda" if torch.cuda.is_available() else "cpu"
            )
            cls._instance.model.to(cls._instance.device)
            cls._instance.model.eval()

        return cls._instance

    @torch.no_grad()
    def score(self, text: str) -> Tuple[Dict[str, float], str]:
        """
        Returns:
            - Dict[str, float]: probability per class
            - str: predicted label
        """

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        outputs = self.model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).squeeze()

        scores = {
            label: round(probs[i].item(), 4)
            for i, label in enumerate(self.LABELS)
        }

        predicted_label = self.LABELS[probs.argmax().item()]

        return scores, predicted_label


In [17]:
def analyze_toxicity(text: str):
    """Analyze toxicity level of text using RoBERTa ToxicityDetector."""
    detector = ToxicityDetector()
    toxicity_scores, predicted_label = detector.score(text)

    return {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "predicted_label": predicted_label,
        "confidence": max(toxicity_scores.values()),
        "toxicity_scores": toxicity_scores
    }


In [18]:
toxic_text = "You're a complete idiot! This is total garbage and anyone who believes it is stupid!"
result = analyze_toxicity(toxic_text)
result

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'text': "You're a complete idiot! This is total garbage and anyone who believes it is stupid!",
 'predicted_label': 'rude',
 'confidence': 0.2178,
 'toxicity_scores': {'friendly': 0.192,
  'neutral': 0.1693,
  'rude': 0.2178,
  'toxic': 0.2134,
  'super_toxic': 0.2075}}

In [ ]:
# %pip install transformers datasets accelerate scikit-learn --quiet

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={
    "train": "train.csv",
    "validation": "val.csv"
})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)


In [ ]:
def encode_labels(batch):
    batch["label"] = label2id[batch["label"]]
    return batch

dataset = dataset.map(encode_labels)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./toxicity_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,  # turn off if no GPU
)


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
trainer.save_model("./roberta-toxicity-5class")
tokenizer.save_pretrained("./roberta-toxicity-5class")


In [ ]:
model_name = "./roberta-toxicity-5class"

In [19]:
text = "You're an idiot and nobody likes you."
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)

probs = outputs.logits.softmax(dim=1)
LABELS[probs.argmax()]

NameError: name 'tokenizer' is not defined